# Network Intrusion Detection — Exploratory Data Analysis & Shift Investigation
**Competition:** GDG on Campus AI/ML Volunteer Recruitment Challenge (Kaggle)
**Task:** Multiclass Network Intrusion Classification (Availability, Benign, Reconnaissance, Spoofing)
**Dataset Attribution:** RT-IoT2022 dataset by B. S. Sharmila and Dr. Rohini Nagapadma, NIE Mysuru.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, f1_score
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

## 1. Load Data and Verify Missingness / Invariants

In [ ]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Train null count:', train.isnull().sum().sum())
print('Test null count:', test.isnull().sum().sum())

## 2. Target Class Distribution (Severe Class Imbalance)

In [ ]:
print('Target Class Counts:')
print(train['Target'].value_counts())
print('\nTarget Class Percentages:')
print((train['Target'].value_counts(normalize=True)*100).round(2))

plt.figure(figsize=(8, 4))
sns.countplot(data=train, x='Target', order=train['Target'].value_counts().index, palette='viridis')
plt.title('Target Class Distribution in Training Set')
plt.ylabel('Count')
plt.show()

## 3. Covariate and Domain Shift (Train vs Test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
proto_comp = pd.DataFrame({
    'Train': train['proto'].value_counts(normalize=True)*100,
    'Test': test['proto'].value_counts(normalize=True)*100
}).fillna(0)
proto_comp.plot(kind='bar', ax=axes[0], color=['#3498db', '#e67e22'])
axes[0].set_title('Protocol Distribution: Train vs Test (%)')
axes[0].set_ylabel('Percentage')

serv_comp = pd.DataFrame({
    'Train': train['service'].value_counts(normalize=True)*100,
    'Test': test['service'].value_counts(normalize=True)*100
}).fillna(0).sort_values('Test', ascending=False).head(6)
serv_comp.plot(kind='bar', ax=axes[1], color=['#3498db', '#e67e22'])
axes[1].set_title('Service Distribution: Train vs Test (%)')
axes[1].set_ylabel('Percentage')
plt.tight_layout()
plt.show()

## 4. Destination Port Specialization (Port 21 Attack vs Diversified Test Ports)

In [ ]:
print('Train top 5 destination ports:')
print(train['id.resp_p'].value_counts().head(5))
print('\nTest top 5 destination ports:')
print(test['id.resp_p'].value_counts().head(5))
print('\nObservation: 79,698 flows in train target port 21 (94.08%), and 10,396 flows in test target port 21 (31.35%).')

## 5. Adversarial Validation
We train a classifier to separate training rows from test rows to evaluate distribution overlap.

In [ ]:
adv_feats = [c for c in test.columns if c not in ['id', 'proto', 'service']]
adv_tr = train[adv_feats].copy(); adv_tr['is_test'] = 0
adv_te = test[adv_feats].copy(); adv_te['is_test'] = 1
adv_df = pd.concat([adv_tr, adv_te], ignore_index=True)

clf = LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1, n_jobs=-1)
clf.fit(adv_df[adv_feats], adv_df['is_test'])
auc = roc_auc_score(adv_df['is_test'], clf.predict_proba(adv_df[adv_feats])[:, 1])
print(f'Adversarial Validation ROC-AUC: {auc:.4f}')

imp = pd.Series(clf.feature_importances_, index=adv_feats).sort_values(ascending=False)
print('\nTop 10 features driving train-test shift:')
print(imp.head(10))